# From one head to a block

Chapters 15–17 supply the hand calculation before these executable checks. The matrices are authored teaching values, not learned semantic features. Execute all cells in a fresh CPU kernel. No network or GPU is used. This notebook reads independent expectations and never rewrites them.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
ROOT = Path.cwd()
assert (ROOT / "src/config/book.mjs").is_file(), "Run from the book repository root"
sys.path.insert(0, str(ROOT / "code/part-iv"))
sys.path.insert(0, str(ROOT / "code/mini-gpt"))
from mechanisms import projected_attention, masked_softmax, two_heads, small_block
import reference
fixture = json.loads((ROOT / "data/part-iv/attention.json").read_text())
attention = projected_attention(*(fixture[key] for key in ("X", "WQ", "WK", "WV")))
for key, expected in fixture["expected"].items():
    np.testing.assert_allclose(attention[key], expected, atol=1e-12, rtol=0)
for key in ("Q", "K", "V", "scores", "weights", "output"):
    print(key, attention[key].shape, attention[key].round(6).tolist())


Q (3, 2) [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
K (3, 2) [[1.0, 1.0], [0.0, 1.0], [1.0, 2.0]]
V (3, 2) [[1.0, 0.0], [1.0, 2.0], [2.0, 2.0]]
scores (3, 3) [[0.707107, 0.0, 0.707107], [0.707107, 0.707107, 1.414214], [1.414214, 0.707107, 2.12132]]
weights (3, 3) [[0.401112, 0.197776, 0.401112], [0.248255, 0.248255, 0.50349], [0.283995, 0.140029, 0.575975]]
output (3, 2) [[1.401112, 1.197776], [1.50349, 1.50349], [1.575975, 1.432009]]


## Exclude a source before normalizing

Visible=True is the mathematical convention in this implementation. A valid query with no visible key raises an error. Padding queries are a separate explicitly excluded case. The first query must reproduce its own value regardless of future values.


In [2]:
visible = np.tril(np.ones((3, 3), dtype=bool))
a = masked_softmax(attention["scores"], visible)
np.testing.assert_allclose(a.sum(axis=-1), 1, atol=1e-12, rtol=0)
assert np.all(a[~visible] == 0)
output = a @ attention["V"]
np.testing.assert_allclose(output[0], [1, 0], atol=1e-12, rtol=0)
try:
    masked_softmax(np.zeros((1, 2)), np.zeros((1, 2), dtype=bool))
except ValueError as error:
    print("Expected rejected valid row:", str(error))
else:
    raise AssertionError("An all-masked valid row was accepted")
print("causal weights", a.round(6).tolist())
print("causal output", output.round(6).tolist())


Expected rejected valid row: a valid query has no visible key
causal weights [[1.0, 0.0, 0.0], [0.5, 0.5, 0.0], [0.283995, 0.140029, 0.575975]]
causal output [[1.0, 0.0], [1.0, 1.0], [1.575975, 1.432009]]


## Independent heads and scalar block arithmetic

The reference path uses Python loops and scalar math, not NumPy matrix products. Equal shapes are supplemented with numerical comparison and the separate future-perturbation tests in the CPU suite.


In [3]:
heads = two_heads(fixture["X"])
x = np.array(fixture["X"]) + np.array([[0, 0], [0, 1], [1, 0]])
block = small_block(x)
expected = reference.block(x.tolist())
np.testing.assert_allclose(block["output"], expected, atol=1e-12, rtol=0)
print("multihead", {key: value.tolist() for key, value in heads.items() if isinstance(value, np.ndarray)})
print("block output", block["output"].round(9).tolist())


multihead {'joined': [[1.0, 0.0], [0.5, 0.7310585786300049], [0.8446375965030364, 0.8446375965030364]], 'output': [[1.0, 1.0], [1.2310585786300048, -0.2310585786300049], [1.6892751930060728, 0.0]]}
block output [[1.999979999, 2.99994], [0.999912048, 1.476742475], [2.999964106, 3.746396089]]
